# 01 — Training (Vimeo-90k V2)
Event-guided video frame interpolation on Vimeo-90k triplet data.
Uses `VimeoTripletDataset` — events generated at native 448x256, no misalignment.

Mount → Config → Data → Visualize → Overfit check → Train → Qualitative results.

In [1]:
# ── Cell 1: Setup & Mount ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/493Project')

%pip install -q torchmetrics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.0 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os
import glob
import random
import torch
from torch.amp import GradScaler
from torch.utils.data import DataLoader

from src.data import VimeoTripletDataset
from src.train import (
    build_model,
    build_criterion,
    build_optimizer,
    build_scheduler,
    load_checkpoint,
    overfit_one_batch,
    run_training,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [3]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
cfg = dict(
    # Paths
    vimeo_root     = '/content/drive/MyDrive/493Project/data/vimeo',
    local_root     = '/content/local_vimeo',
    checkpoint_dir = '/content/drive/MyDrive/493Project/checkpoints/vimeo_v6_dual',

    # Model
    model_type     = 'dual',       # 'dual' | 'baseline'

    # Data
    patch_size     = 256,
    batch_size     = 16,
    num_workers    = 4,

    # Optimizer
    lr             = 2e-4,
    lr_min         = 1e-6,
    weight_decay   = 0.0,

    # Scheduler
    warmup_epochs  = 0,

    # Loss — Charbonnier only (lambda guards now skip VGG/SSIM computation)
    lambda_char            = 1.0,
    lambda_perceptual      = 0.0,
    lambda_ssim            = 0.0,
    lambda_event_weighted  = 0.0,

    # Training
    num_epochs     = 40,
    max_norm       = 1.0,
)

os.makedirs(cfg['checkpoint_dir'], exist_ok=True)

In [4]:
# ── Cell 4: Load data — extract tar to local SSD ────────────────────────────
import subprocess

vimeo_root = cfg['vimeo_root']
local_root = cfg['local_root']
os.makedirs(local_root, exist_ok=True)

tar_on_drive = os.path.join(vimeo_root, 'processed.tar')
marker = os.path.join(local_root, '.copy_done')

if not os.path.exists(marker):
    assert os.path.exists(tar_on_drive), \
        f'processed.tar not found on Drive — run 00b_tar_processed.ipynb first'

    subprocess.run(['apt-get', 'install', '-qq', '-y', 'pv'], capture_output=True)
    total_bytes = os.path.getsize(tar_on_drive)
    print(f'Extracting processed.tar ({total_bytes / 1e9:.1f} GB) to local SSD...')
    subprocess.run(
        f'pv -f -s {total_bytes} "{tar_on_drive}" | tar xf - -C "{local_root}"',
        shell=True, check=True,
    )
    open(marker, 'w').close()
    print('Done.')
else:
    print('Local data already exists, skipping extraction.')

# Load split lists and resolve to local paths
def load_split(split_file):
    path = os.path.join(vimeo_root, split_file)
    with open(path) as f:
        entries = [line.strip() for line in f if line.strip()]
    dirs = []
    for rel in entries:
        d = os.path.join(local_root, rel)
        if os.path.isdir(d):
            dirs.append((rel, d))
    return dirs

train_entries = load_split('tri_trainlist.txt')
val_entries   = load_split('tri_vallist.txt')

# If val has no processed triplets, split from train (90/10)
if len(val_entries) == 0 and len(train_entries) > 0:
    print(f'No processed val triplets found — splitting 90/10 from {len(train_entries)} train entries')
    rng = random.Random(42)
    all_entries = list(train_entries)
    rng.shuffle(all_entries)
    n_val = max(1, int(len(all_entries) * 0.10))
    val_entries   = all_entries[:n_val]
    train_entries = all_entries[n_val:]

train_dirs = [d for _, d in train_entries]
val_dirs   = [d for _, d in val_entries]
print(f'Train: {len(train_dirs)} | Val: {len(val_dirs)}')

Extracting processed.tar (5.3 GB) to local SSD...
Done.
No processed val triplets found — splitting 90/10 from 1900 train entries
Train: 1710 | Val: 190


In [5]:
# ── Cell 5: Visualize event voxels — sanity check ────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

sample_dirs = train_dirs[:4]
fig, axes = plt.subplots(len(sample_dirs), 4, figsize=(20, 5 * len(sample_dirs)))
if len(sample_dirs) == 1:
    axes = axes[np.newaxis, :]

col_titles = ['Frame f0 (im1)', 'Frame f1 (im3)', 'Ground Truth (im2)', 'Event Voxel (sum of bins)']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=13, fontweight='bold')

for row, d in enumerate(sample_dirs):
    f0  = Image.open(os.path.join(d, 'im1.png')).convert('RGB')
    f1  = Image.open(os.path.join(d, 'im3.png')).convert('RGB')
    gt  = Image.open(os.path.join(d, 'im2.png')).convert('RGB')
    voxel = torch.load(os.path.join(d, 'voxel.pt'), weights_only=True)

    axes[row, 0].imshow(f0); axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    axes[row, 1].imshow(f1); axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])
    axes[row, 2].imshow(gt); axes[row, 2].set_xticks([]); axes[row, 2].set_yticks([])

    evt_sum = voxel.sum(dim=0).numpy()
    abs_max = max(abs(evt_sum.min()), abs(evt_sum.max()), 1e-8)
    axes[row, 3].imshow(evt_sum, cmap='RdBu_r', vmin=-abs_max, vmax=abs_max)
    axes[row, 3].set_xticks([]); axes[row, 3].set_yticks([])

    density = (voxel.abs() > 0.01).float().mean().item() * 100
    axes[row, 0].set_ylabel(
        f'voxel: {tuple(voxel.shape)}\n'
        f'range: [{voxel.min():.2f}, {voxel.max():.2f}]\n'
        f'active: {density:.1f}%',
        fontsize=8
    )

plt.suptitle('Vimeo-90k Event Voxels — Do events encode motion?', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [6]:
# ── Cell 6: Create datasets and dataloaders ──────────────────────────────────
train_dataset = VimeoTripletDataset(train_dirs, patch_size=cfg['patch_size'], augment=True)
val_dataset   = VimeoTripletDataset(val_dirs,   patch_size=cfg['patch_size'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg['batch_size'], shuffle=False,
                          num_workers=cfg['num_workers'], pin_memory=True)

print(f'Train: {len(train_dataset)} clips, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} clips, {len(val_loader)} batches')

# Quick shape check
f0, f1, voxel, gt = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  f0:    {tuple(f0.shape)}')
print(f'  f1:    {tuple(f1.shape)}')
print(f'  voxel: {tuple(voxel.shape)}')
print(f'  gt:    {tuple(gt.shape)}')

Train: 1710 clips, 107 batches
Val:   190 clips, 12 batches

Batch shapes:
  f0:    (16, 3, 256, 256)
  f1:    (16, 3, 256, 256)
  voxel: (16, 5, 256, 256)
  gt:    (16, 3, 256, 256)


In [7]:
# ── Cell 7: Model, optimizer, scheduler, loss ────────────────────────────────
model     = build_model(cfg, device)
optimizer = build_optimizer(model, cfg)
scheduler = build_scheduler(optimizer, cfg)
scaler    = GradScaler('cuda', enabled=False)  # AMP disabled — re-enable once training is stable
criterion = build_criterion(cfg, device)

# Resume from checkpoint if one exists
start_epoch, best_psnr = load_checkpoint(
    cfg['checkpoint_dir'], model, optimizer, scheduler, device
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {cfg["model_type"]} | Parameters: {n_params:,}')
print(f'LR: {cfg["lr"]} | Warmup: {cfg.get("warmup_epochs", 0)} epochs | AMP: {scaler.is_enabled()}')
print(f'Loss: Charb({cfg["lambda_char"]}) + Perc({cfg["lambda_perceptual"]})')

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 230MB/s]


Model: dual | Parameters: 27,894,531
LR: 0.0002 | Warmup: 0 epochs | AMP: False
Loss: Charb(1.0) + Perc(0.0)


In [8]:
# ── Cell 8: Sanity check — overfit one batch ─────────────────────────────────
from src.losses import CharbonnierLoss

overfit_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                            num_workers=0, pin_memory=True)
sane_model     = build_model(cfg, device)
sane_optimizer = torch.optim.Adam(sane_model.parameters(), lr=2e-3)
sane_criterion = CharbonnierLoss()

final_loss = overfit_one_batch(
    sane_model, overfit_loader, sane_criterion, sane_optimizer, device, iters=1000
)
print(f'Overfit final loss: {final_loss:.6f}  (target < 0.01)')
del sane_model, sane_optimizer, sane_criterion, overfit_loader

Overfitting batch:   4%|▎         | 35/1000 [00:06<03:00,  5.35it/s, loss=0.043388]


KeyboardInterrupt: 

In [43]:
# ── Cell 8b: DIAGNOSTIC — Find the NaN source ───────────────────────────────
model.train()
f0, f1, voxel, gt = next(iter(train_loader))
f0, f1, voxel, gt = f0.to(device), f1.to(device), voxel.to(device), gt.to(device)

print('Input ranges:')
print(f'  f0:    [{f0.min():.3f}, {f0.max():.3f}]')
print(f'  f1:    [{f1.min():.3f}, {f1.max():.3f}]')
print(f'  voxel: [{voxel.min():.3f}, {voxel.max():.3f}], std={voxel.std():.3f}')

rgb = torch.cat([f0, f1], dim=1)
mean_frame = (f0 + f1) / 2

# Check each stage
with torch.no_grad():
    r1 = model.inc_rgb(rgb);   e1 = model.inc_evt(voxel)
    print(f'  r1: [{r1.min():.3f}, {r1.max():.3f}], nan={r1.isnan().any()}')
    print(f'  e1: [{e1.min():.3f}, {e1.max():.3f}], nan={e1.isnan().any()}')
    r5 = model.down4_rgb(model.down3_rgb(model.down2_rgb(model.down1_rgb(r1))))
    e5 = model.down4_evt(model.down3_evt(model.down2_evt(model.down1_evt(e1))))
    print(f'  r5: [{r5.min():.3f}, {r5.max():.3f}], nan={r5.isnan().any()}')
    print(f'  e5: [{e5.min():.3f}, {e5.max():.3f}], nan={e5.isnan().any()}')
    pred = model(rgb, voxel)
    print(f'  pred: [{pred.min():.3f}, {pred.max():.3f}], nan={pred.isnan().any()}')
    print(f'  mean_frame PSNR: {-10*torch.log10(((mean_frame-gt)**2).mean()):.2f} dB')

# Now do ONE training step and check
model.train()
optimizer.zero_grad()
pred = model(rgb, voxel)
loss = criterion(pred, gt)
print(f'\nStep 0: loss={loss.item():.4f}, pred nan={pred.isnan().any()}')
loss.backward()
grad_norm = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None)**0.5
print(f'  grad_norm={grad_norm:.4f}')
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

# Second step
optimizer.zero_grad()
pred = model(rgb, voxel)
loss = criterion(pred, gt)
print(f'Step 1: loss={loss.item():.4f}, pred nan={pred.isnan().any()}')

Input ranges:
  f0:    [0.000, 1.000]
  f1:    [0.000, 1.000]
  voxel: [-1.695, 65.341], std=0.329
  r1: [nan, nan], nan=True
  e1: [nan, nan], nan=True
  r5: [nan, nan], nan=True
  e5: [nan, nan], nan=True
  pred: [nan, nan], nan=True
  mean_frame PSNR: 23.95 dB

Step 0: loss=nan, pred nan=True
  grad_norm=nan
Step 1: loss=nan, pred nan=True


In [9]:
# ── Cell 9: Full training loop ───────────────────────────────────────────────
training_log = run_training(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler, scaler,
    device, cfg,
    start_epoch=start_epoch,
    best_psnr=best_psnr,
)
print('Training complete.')

Epoch 1: 100%|██████████| 107/107 [01:03<00:00,  1.69it/s, loss=0.0380, gnorm=0.7]


Epoch 001/40 | Train 0.0419 | Val 0.0340 | PSNR 23.21 dB | SSIM 0.7348
  -> New best PSNR: 23.21 dB


Epoch 2: 100%|██████████| 107/107 [01:04<00:00,  1.66it/s, loss=0.0344, gnorm=0.2]


Epoch 002/40 | Train 0.0321 | Val 0.0326 | PSNR 23.55 dB | SSIM 0.7465
  -> New best PSNR: 23.55 dB


Epoch 3: 100%|██████████| 107/107 [01:04<00:00,  1.66it/s, loss=0.0306, gnorm=0.4]


...

Epoch 038/40 | Train 0.0183 | Val 0.0189 | PSNR 28.44 dB | SSIM 0.8777


Epoch 39: 100%|██████████| 107/107 [01:04<00:00,  1.66it/s, loss=0.0215, gnorm=0.1]


Epoch 039/40 | Train 0.0183 | Val 0.0192 | PSNR 28.31 dB | SSIM 0.8764


Epoch 40: 100%|██████████| 107/107 [01:04<00:00,  1.66it/s, loss=0.0182, gnorm=0.1]


Epoch 040/40 | Train 0.0182 | Val 0.0186 | PSNR 28.60 dB | SSIM 0.8785
  -> New best PSNR: 28.60 dB
Training complete.


In [10]:
# ── Cell 10: Qualitative results ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image
from src.train import _forward

best_model = build_model(cfg, device)
best_model.load_state_dict(
    torch.load(os.path.join(cfg['checkpoint_dir'], 'best_model.pth'), map_location=device)
)
best_model.eval()

def load_vimeo_clip(clip_dir, patch_size=256):
    f0  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im1.png')).convert('RGB'))
    f1  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im3.png')).convert('RGB'))
    gt  = TF.to_tensor(Image.open(os.path.join(clip_dir, 'im2.png')).convert('RGB'))
    evt = torch.load(os.path.join(clip_dir, 'voxel.pt'), weights_only=True)
    # Voxel already normalized during data prep — no double normalization
    _, H, W = f0.shape
    top  = (H - patch_size) // 2
    left = (W - patch_size) // 2
    f0  = TF.crop(f0,  top, left, patch_size, patch_size)
    f1  = TF.crop(f1,  top, left, patch_size, patch_size)
    gt  = TF.crop(gt,  top, left, patch_size, patch_size)
    evt = evt[:, top:top+patch_size, left:left+patch_size]
    return f0, f1, gt, evt

show_dirs = val_dirs[:6]
fig, axes = plt.subplots(len(show_dirs), 5, figsize=(22, 4.5 * len(show_dirs)))
if len(show_dirs) == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(['Frame f0', 'Frame f1', 'Ground Truth', 'Prediction', 'Error (x4)']):
    axes[0, col].set_title(title, fontsize=13, fontweight='bold')

with torch.no_grad():
    for row, clip_dir in enumerate(show_dirs):
        f0, f1, gt, evt = load_vimeo_clip(clip_dir)
        f0_d = f0.unsqueeze(0).to(device)
        f1_d = f1.unsqueeze(0).to(device)
        evt_d = evt.unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            outputs = _forward(best_model, f0_d, f1_d, evt_d)
        pred = outputs[0].squeeze(0).float().cpu().clamp(0, 1)

        error = (gt - pred).abs() * 4
        mse   = ((gt - pred) ** 2).mean().item()
        psnr  = -10 * np.log10(mse + 1e-10)

        for col, img in enumerate([f0, f1, gt, pred, error]):
            axes[row, col].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        axes[row, 3].set_xlabel(f'PSNR: {psnr:.2f} dB', fontsize=9, color='steelblue')

plt.suptitle('Vimeo-90k — Qualitative Results on Validation', fontsize=15, y=1.01)
plt.tight_layout()
fig_path = os.path.join(cfg['checkpoint_dir'], 'qualitative_results.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved to {fig_path}')

Output hidden; open in https://colab.research.google.com to view.